# LSTM — `direction_5day` (lb20)  ·  binary up/down classifier

Config-driven LSTM **classification** on the **GPU**. Same framework as the
`return_5day` regressor, but the target is the binary 1/0 label `direction_5day`
(1 = the forward 5-day return is positive). The model emits a raw **logit**; it is
trained with `BCEWithLogitsLoss` (sigmoid applied inside the loss) and evaluated
with `classification_metrics` — no target scaling, no inverse-transform.

- **Data is referenced, not cloned**: tensors read from `src/train_test_set/<dataset>/`
  (built by `train_test_creator.ipynb` with **`SCALE_TARGET=False`**); the run records
  only the dataset name + content **hash**.
- **Sample**: a `(lookback=20, n_features)` window; **label** = `direction_5day` at the
  window's last day.
- Direction-skill metrics land in the SAME `index.csv` columns (`*_dir_accuracy`,
  `*_dir_auc`) the return regressor fills, so classifier vs regressor compare directly.

Edit `configs/direction_5day_lb20.yaml` (or `CONFIG_PATH` below) and **Run All**.

## Setup — config + framework imports

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml

sys.path.insert(0, os.path.abspath(".."))       # src/model  -> `common`
sys.path.insert(0, os.path.abspath("."))          # src/model/lstm -> `model.py`

from common import (
    RunDir, Trainer, TrainConfig, load_dataset, classification_metrics,
    resolve_device, set_seed, to_loaders, append_run,
)
import model as lstm_model

CONFIG_PATH = os.environ.get("CONFIG_PATH", "configs/direction_5day_lb20.yaml")
with open(CONFIG_PATH) as f:
    CONFIG = yaml.safe_load(f)

RUNS_DIR = os.path.abspath("../runs")
set_seed(CONFIG.get("seed", 42))
DEVICE = resolve_device(CONFIG.get("device", "auto"))
gpu = f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""
print(f"Config : {CONFIG_PATH}")
print(f"Dataset: {CONFIG['dataset']}")
print(f"Task   : {CONFIG.get('task', 'classification')}")
print(f"Device : {DEVICE}{gpu}")
CONFIG

## Load the referenced dataset (verify hash)

In [ ]:
dataset = load_dataset(CONFIG["dataset"])
assert dataset.target_scaler is None, (
    "direction_5day is a 0/1 label; rebuild the dataset with SCALE_TARGET=False")
base_rate = float(np.mean(dataset.y_train))
print(f"dataset : {dataset.name}")
print(f"hash    : {dataset.hash}")
print(f"X_train {dataset.X_train.shape} | X_val {dataset.X_val.shape} | X_test {dataset.X_test.shape}")
print(f"lookback={dataset.lookback}  n_features={dataset.n_features}  target={dataset.meta.get('target')}")
print(f"train base rate (share of up days): {base_rate:.3f}")
pd.DataFrame(dataset.reference()["shapes"]).T.rename(columns={0: "windows", 1: "lookback", 2: "n_features"})

## Create the run folder & build the model

In [ ]:
run = RunDir.create(base_dir=RUNS_DIR, run_name=CONFIG["run_name"], config=CONFIG)
run.update_metadata(dataset=dataset.reference(), device=str(DEVICE), task="classification")

mc = CONFIG["model"]
arch = lstm_model.arch_dict(
    n_features=dataset.n_features,
    hidden_size=mc["hidden_size"], num_layers=mc["num_layers"], dropout=mc["dropout"],
)
net = lstm_model.build_model(**arch["kwargs"])   # last Linear -> raw logit (no sigmoid)
n_params = sum(p.numel() for p in net.parameters())
print(f"run folder: {run.dir}")
print(net)
print(f"parameters: {n_params:,}")

## Train on GPU (BCEWithLogitsLoss, early stopping, TensorBoard)

Best epoch = lowest **val BCE**. The same `Trainer` is used as for regression — only
the loss is swapped (the model output is a logit; the sigmoid lives inside the loss).

In [ ]:
tcfg = TrainConfig.from_dict(CONFIG.get("train", {}))
train_loader, val_loader, test_loader = to_loaders(dataset, tcfg.batch_size, DEVICE)

trainer = Trainer(net, tcfg, run, DEVICE, arch=arch, criterion=nn.BCEWithLogitsLoss())
history = trainer.fit(train_loader, val_loader)

pd.DataFrame(history).to_csv(os.path.join(run.results_dir, "loss_history.csv"), index_label="epoch")
print(f"TensorBoard event files -> {run.tb_dir}")

## Evaluate on val / test (classification)

Predictions are logits -> `sigmoid` -> P(up). `majority_baseline_acc` = accuracy of
always predicting the majority class; `dir_auc` (ROC-AUC of the probability) is the
threshold-free skill and is directly comparable to the return regressor's `dir_auc`.

In [ ]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.asarray(z, dtype=float)))

results = {}
preds = {}
for name, loader, y_true, dts in [
    ("val", val_loader, dataset.y_val, dataset.dates_val),
    ("test", test_loader, dataset.y_test, dataset.dates_test),
]:
    y_prob = sigmoid(trainer.predict(loader))
    y_true = np.asarray(y_true, dtype=float).ravel()
    results[name] = classification_metrics(y_true, y_prob)
    df_p = pd.DataFrame({"y_true": y_true.astype(int), "y_prob": y_prob})
    if dts is not None:
        df_p.insert(0, "date", dts.astype(str))
    df_p.to_csv(os.path.join(run.results_dir, f"predictions_{name}.csv"), index=False)
    preds[name] = df_p

trainer.log_hparams(
    {"hidden_size": mc["hidden_size"], "num_layers": mc["num_layers"],
     "dropout": mc["dropout"], "lr": tcfg.lr, "batch_size": tcfg.batch_size},
    {f"test_{k}": v for k, v in results["test"].items() if isinstance(v, (int, float)) and not isinstance(v, bool)},
)

metrics_df = pd.DataFrame(results).T
metrics_df

## Plots — BCE loss curve & test ROC (saved to results/)

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(history["train"], label="train")
axes[0].plot(history["val"], label="val")
axes[0].axvline(trainer.best_epoch - 1, color="grey", ls="--", lw=1,
                label=f"best epoch {trainer.best_epoch}")
axes[0].set_title("BCE loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

tp = preds["test"]
fpr, tpr, _ = roc_curve(tp["y_true"], tp["y_prob"])
axes[1].plot(fpr, tpr, lw=1.5, label=f"AUC = {results['test']['dir_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], color="grey", lw=0.5, ls="--")
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title(f"Test ROC — {CONFIG['dataset']}"); axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(run.results_dir, "loss_and_roc.png"), dpi=130, bbox_inches="tight")
plt.show()

## Finalize — write metrics + registry row

In [ ]:
with open(os.path.join(run.results_dir, "metrics.json"), "w") as f:
    json.dump(results, f, indent=2)

run.update_metadata(
    model={"type": "LSTM", **arch["kwargs"], "n_params": int(n_params)},
    training={"best_epoch": int(trainer.best_epoch),
              "best_val_loss": float(trainer.best_val),
              **{k: getattr(tcfg, k) for k in ("batch_size", "lr", "weight_decay",
                                               "max_epochs", "patience")}},
    metrics=results,
)
trainer.close()

append_run(RUNS_DIR, {
    "run_id": run.run_id,
    "created_at": run.metadata["created_at"],
    "dataset_name": dataset.name,
    "dataset_hash": dataset.hash,
    "model_type": "LSTM",
    "task": "classification",
    "lookback": dataset.lookback,
    "n_features": dataset.n_features,
    "best_epoch": trainer.best_epoch,
    "best_val_loss": round(trainer.best_val, 6),
    "val_dir_accuracy": round(results["val"]["dir_accuracy"], 4),
    "val_dir_auc": round(results["val"]["dir_auc"], 4),
    "val_pr_auc": round(results["val"]["pr_auc"], 4),
    "val_f1": round(results["val"]["f1"], 4),
    "test_dir_accuracy": round(results["test"]["dir_accuracy"], 4),
    "test_dir_auc": round(results["test"]["dir_auc"], 4),
    "test_pr_auc": round(results["test"]["pr_auc"], 4),
    "test_f1": round(results["test"]["f1"], 4),
    "test_log_loss": round(results["test"]["log_loss"], 6),
    "test_base_rate": round(results["test"]["base_rate"], 4),
    "test_beats_majority": results["test"]["beats_majority"],
    "git_sha": run.metadata["git_sha"],
    "run_dir": os.path.relpath(run.dir, os.path.abspath("..")),
})

print(f"Run complete -> {run.dir}")
print(f"test: acc={results['test']['dir_accuracy']:.3f} (majority {results['test']['majority_baseline_acc']:.3f}) "
      f"| AUC={results['test']['dir_auc']:.3f} | beats_majority={results['test']['beats_majority']}")
print(f"Compare runs:  tensorboard --logdir {RUNS_DIR}")